In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import lightgbm as lgb
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Load segments data
df = pd.read_csv("/Volumes/MacData/KWF/NLP Analysis/final data/df_NLP.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 769365 entries, 0 to 769364
Data columns (total 90 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   recorder_id             769365 non-null  object 
 1   original_clip_name      769365 non-null  object 
 2   segment_id              769365 non-null  int64  
 3   start_time              769365 non-null  float64
 4   end_time                769365 non-null  float64
 5   window_duration         769365 non-null  float64
 6   onset_confidence        769365 non-null  int64  
 7   has_onset               769365 non-null  bool   
 8   preliminary_rms         769365 non-null  float64
 9   RMS Energy              769365 non-null  float64
 10  Zero Crossing Rate      769365 non-null  float64
 11  Spectral Bandwidth      769365 non-null  float64
 12  Spectral Rolloff (85%)  769365 non-null  float64
 13  Spectral Flatness       769365 non-null  float64
 14  Spectral Contrast   

In [3]:
df.sample(10)

,recorder_id,original_clip_name,segment_id,start_time,end_time,window_duration,onset_confidence,has_onset,preliminary_rms,RMS Energy,...,Precipitation,Humidity,Weathercode,Weather Desc,Timestamp Local,Sunrise,Sunset,Time Of Day,Timestamp UTC,Datetime_hour
321475,Audio_Moth_1,Audio_Moth_1_20250320_023306.wav,0,0.00,0.60,0.6,8,True,0.070776,0.070776,...,0.0,94,1,Mainly clear,2025-03-20 03:00:00-06:00,2025-03-20 05:38:44.718176-06:00,2025-03-20 17:45:09.530345-06:00,night,2025-03-20 09:00:00+00:00,3
503302,Audio_Moth_2,Audio_Moth_2_20250321_042050.wav,2,0.72,1.32,0.6,11,True,0.109456,0.109456,...,0.0,85,3,Overcast,2025-03-21 04:00:00-06:00,2025-03-21 05:38:12.435284-06:00,2025-03-21 17:45:06.084924-06:00,dawn,2025-03-21 10:00:00+00:00,4
80404,Audio_Moth_1,Audio_Moth_1_20250319_053357.wav,4,1.44,2.04,0.6,7,True,0.014249,0.014249,...,0.0,96,0,Clear sky,2025-03-19 06:00:00-06:00,2025-03-19 05:39:16.886691-06:00,2025-03-19 17:45:12.872250-06:00,morning,2025-03-19 12:00:00+00:00,6
252792,Audio_Moth_1,Audio_Moth_1_20250320_042448.wav,1,0.36,0.96,0.6,7,True,0.033682,0.033682,...,0.0,94,0,Clear sky,2025-03-20 04:00:00-06:00,2025-03-20 05:38:44.718176-06:00,2025-03-20 17:45:09.530345-06:00,dawn,2025-03-20 10:00:00+00:00,4
531347,Audio_Moth_2,Audio_Moth_2_20250320_001808.wav,5,1.80,2.40,0.6,11,True,0.098472,0.098472,...,0.0,87,1,Mainly clear,2025-03-20 00:00:00-06:00,2025-03-20 05:38:44.718176-06:00,2025-03-20 17:45:09.530345-06:00,night,2025-03-20 06:00:00+00:00,0
60923,Audio_Moth_1,Audio_Moth_1_20250321_004245.wav,2,0.72,1.32,0.6,9,True,0.100269,0.100269,...,0.0,87,1,Mainly clear,2025-03-21 01:00:00-06:00,2025-03-21 05:38:12.435284-06:00,2025-03-21 17:45:06.084924-06:00,night,2025-03-21 07:00:00+00:00,1
586067,Audio_Moth_5,Audio_Moth_5_20250318_033135.wav,6,2.16,2.76,0.6,6,True,0.071825,0.071825,...,0.0,88,3,Overcast,2025-03-18 04:00:00-06:00,2025-03-18 05:39:48.918172-06:00,2025-03-18 17:45:16.092609-06:00,dawn,2025-03-18 10:00:00+00:00,4
747514,Audio_Moth_6,Audio_Moth_6_20250317_231026.wav,3,1.08,1.68,0.6,6,True,0.119973,0.119973,...,0.0,92,3,Overcast,2025-03-17 23:00:00-06:00,2025-03-17 05:40:20.789893-06:00,2025-03-17 17:45:19.173328-06:00,night,2025-03-18 05:00:00+00:00,23
721436,Audio_Moth_6,Audio_Moth_6_20250318_022341.wav,0,0.00,0.60,0.6,10,True,0.147926,0.147926,...,0.0,89,0,Clear sky,2025-03-18 02:00:00-06:00,2025-03-18 05:39:48.918172-06:00,2025-03-18 17:45:16.092609-06:00,night,2025-03-18 08:00:00+00:00,2
453713,Audio_Moth_2,Audio_Moth_2_20250318_100728.wav,1,0.36,0.96,0.6,11,True,0.014906,0.014906,...,0.2,67,51,Light drizzle,2025-03-18 10:00:00-06:00,2025-03-18 05:39:48.918172-06:00,2025-03-18 17:45:16.092609-06:00,morning,2025-03-18 16:00:00+00:00,10


In [4]:
df['Sim Type'].unique()

array([nan, 'Human Presence on Trail', 'Vehicle', 'Gunshot', 'Chainsaw',
       'Human Presence off Trail'], dtype=object)

In [11]:
core_features = [
    'RMS Energy', 'Spectral Contrast', 'Spectral Flatness',
    'Spectral Bandwidth', 'Spectral Rolloff (85%)',
    'Onset Strength', 'MFCC_8', 'MFCC_9', 'MFCC_12', 'MFCC_13'
]

df['is_simulation'] = (~df['Sim Type'].isna()).astype(int)

print(f"Total samples: {len(df)}")
print(f"Simulation samples: {df['is_simulation'].sum()}")
print(f"Baseline samples: {(df['is_simulation'] == 0).sum()}")
print(f"Simulation types:\n{df['Sim Type'].value_counts()}")

Total samples: 769365
Simulation samples: 6657
Baseline samples: 762708
Simulation types:
Sim Type
Human Presence on Trail     5425
Vehicle                      812
Gunshot                      196
Chainsaw                     175
Human Presence off Trail      49
Name: count, dtype: int64


In [16]:
X = df[core_features].copy()
y = df['is_simulation'].copy()
print(f"Missing values in features: {X.isnull().sum().sum()}")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set - Simulation: {y_train.sum()}, Baseline: {(y_train == 0).sum()}")
print(f"Test set - Simulation: {y_test.sum()}, Baseline: {(y_test == 0).sum()}")


Missing values in features: 0
Train set - Simulation: 5326, Baseline: 610166
Test set - Simulation: 1331, Baseline: 152542


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scale_pos_weight = (y_train == 0).sum() / y_train.sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

Scale pos weight: 114.56


In [ ]:
xgb_model = xgb.XGBClassifier(
    random_state=42,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=20,
    eval_metric='logloss',
    reg_alpha=1,
    reg_lambda=1
)

xgb_model.fit(X_train_scaled, y_train)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

print(f"XGBoost ROC-AUC: {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.1, 0.9, 0.05)
best_f1 = 0
best_threshold = 0.5
results = []

for thresh in thresholds:
    y_pred_thresh = (y_pred_proba_xgb >= thresh).astype(int)
    precision = precision_score(y_test, y_pred_thresh)
    recall = recall_score(y_test, y_pred_thresh)
    f1 = f1_score(y_test, y_pred_thresh)
    
    results.append([thresh, precision, recall, f1])
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh

results_df = pd.DataFrame(results, columns=['Threshold', 'Precision', 'Recall', 'F1'])
print(f"\nBest threshold: {best_threshold:.2f} with F1: {best_f1:.3f}")

print("\nTop 5 thresholds by F1 score:")
print(results_df.nlargest(5, 'F1')[['Threshold', 'Precision', 'Recall', 'F1']].round(3))

y_pred_xgb = (y_pred_proba_xgb >= best_threshold).astype(int)
print(f"\nFinal XGBoost Results (threshold={best_threshold:.2f}):")
print(classification_report(y_test, y_pred_xgb))

precision_recall_balance = results_df.copy()
precision_recall_balance['PR_diff'] = abs(precision_recall_balance['Precision'] - precision_recall_balance['Recall'])
best_balanced_idx = precision_recall_balance.loc[precision_recall_balance['F1'] > 0.08].nsmallest(1, 'PR_diff').index[0]
balanced_threshold = precision_recall_balance.loc[best_balanced_idx, 'Threshold']

y_pred_balanced = (y_pred_proba_xgb >= balanced_threshold).astype(int)
print(f"\nBalanced Precision-Recall Results (threshold={balanced_threshold:.2f}):")
print(classification_report(y_test, y_pred_balanced))

XGBoost ROC-AUC: 0.9439

Best threshold: 0.60 with F1: 0.260

Top 5 thresholds by F1 score:
    Threshold  Precision  Recall     F1
10       0.60      0.290   0.235  0.260
9        0.55      0.207   0.325  0.253
11       0.65      0.406   0.183  0.253
8        0.50      0.155   0.427  0.228
12       0.70      0.465   0.143  0.218

Final XGBoost Results (threshold=0.60):
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    152542
           1       0.29      0.24      0.26      1331

    accuracy                           0.99    153873
   macro avg       0.64      0.62      0.63    153873
weighted avg       0.99      0.99      0.99    153873


Balanced Precision-Recall Results (threshold=0.60):
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    152542
           1       0.29      0.24      0.26      1331

    accuracy                           0.99    153873
   macro avg       0.64      

In [24]:
# LightGBM model
lgb_model = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    reg_alpha=1,
    reg_lambda=1,
    verbose=-1
)

lgb_model.fit(X_train_scaled, y_train)
y_pred_proba_lgb = lgb_model.predict_proba(X_test_scaled)[:, 1]

print(f"LightGBM ROC-AUC: {roc_auc_score(y_test, y_pred_proba_lgb):.4f}")

# Best threshold by F1
thresholds = np.arange(0.1, 0.8, 0.05)
best_f1_lgb = 0
best_thresh_lgb = 0.5

for thresh in thresholds:
    y_pred = (y_pred_proba_lgb >= thresh).astype(int)
    f1 = f1_score(y_test, y_pred)
    if f1 > best_f1_lgb:
        best_f1_lgb = f1
        best_thresh_lgb = thresh

y_pred_lgb = (y_pred_proba_lgb >= best_thresh_lgb).astype(int)
print(f"Best threshold: {best_thresh_lgb:.2f}, F1: {best_f1_lgb:.3f}")
print(classification_report(y_test, y_pred_lgb))

LightGBM ROC-AUC: 0.9413
Best threshold: 0.75, F1: 0.134
              precision    recall  f1-score   support

           0       1.00      0.92      0.96    152542
           1       0.07      0.70      0.13      1331

    accuracy                           0.92    153873
   macro avg       0.54      0.81      0.55    153873
weighted avg       0.99      0.92      0.95    153873



In [25]:
# Percentile approach comparison
percentiles = [0.1, 0.2, 0.5, 1.0, 1.5, 2.0]

print("XGBoost percentile results:")
for pct in percentiles:
    n_top = max(1, int(len(y_test) * pct / 100))
    top_indices = y_pred_proba_xgb.argsort()[-n_top:]
    y_pred_top = np.zeros_like(y_test)
    y_pred_top[top_indices] = 1
    
    precision = precision_score(y_test, y_pred_top)
    recall = recall_score(y_test, y_pred_top)
    f1 = f1_score(y_test, y_pred_top)
    print(f"Top {pct}%: P={precision:.3f}, R={recall:.3f}, F1={f1:.3f}")

print("\nLightGBM percentile results:")
for pct in percentiles:
    n_top = max(1, int(len(y_test) * pct / 100))
    top_indices = y_pred_proba_lgb.argsort()[-n_top:]
    y_pred_top = np.zeros_like(y_test)
    y_pred_top[top_indices] = 1
    
    precision = precision_score(y_test, y_pred_top)
    recall = recall_score(y_test, y_pred_top)
    f1 = f1_score(y_test, y_pred_top)
    print(f"Top {pct}%: P={precision:.3f}, R={recall:.3f}, F1={f1:.3f}")

# Top predictions analysis
test_df = pd.DataFrame({
    'true_label': y_test.values,
    'xgb_proba': y_pred_proba_xgb,
    'lgb_proba': y_pred_proba_lgb,
    'sim_type': df.loc[y_test.index, 'Sim Type'].values
})

print(f"\nXGBoost top 20 precision: {test_df.nlargest(20, 'xgb_proba')['true_label'].mean():.3f}")
print(f"LightGBM top 20 precision: {test_df.nlargest(20, 'lgb_proba')['true_label'].mean():.3f}")

XGBoost percentile results:
Top 0.1%: P=0.582, R=0.067, F1=0.120
Top 0.2%: P=0.495, R=0.114, F1=0.186
Top 0.5%: P=0.358, R=0.207, F1=0.262
Top 1.0%: P=0.247, R=0.285, F1=0.265
Top 1.5%: P=0.196, R=0.340, F1=0.248
Top 2.0%: P=0.167, R=0.385, F1=0.233

LightGBM percentile results:
Top 0.1%: P=0.523, R=0.060, F1=0.108
Top 0.2%: P=0.427, R=0.098, F1=0.160
Top 0.5%: P=0.286, R=0.165, F1=0.210
Top 1.0%: P=0.211, R=0.243, F1=0.226
Top 1.5%: P=0.175, R=0.304, F1=0.222
Top 2.0%: P=0.151, R=0.349, F1=0.211

XGBoost top 20 precision: 0.850
LightGBM top 20 precision: 0.650


In [26]:
sentinel_species = [
    'Arremon aurantiirostris_Orange-billed Sparrow',
    'Hylopezus perspicillatus_Streak-chested Antpitta', 
    'Myiornis atricapillus_Black-capped Pygmy-Tyrant'
]

print("Sentinel species counts:")
for species in sentinel_species:
    count = (df['species'] == species).sum()
    print(f"{species}: {count}")

print(f"\nTotal species in dataset: {df['species'].nunique()}")
print("\nTop 10 most common species:")
print(df['species'].value_counts().head(10))

Sentinel species counts:
Arremon aurantiirostris_Orange-billed Sparrow: 13979
Hylopezus perspicillatus_Streak-chested Antpitta: 7567
Myiornis atricapillus_Black-capped Pygmy-Tyrant: 32097

Total species in dataset: 205

Top 10 most common species:
species
Ciccaba nigrolineata_Black-and-white Owl           174986
Pulsatrix perspicillata_Spectacled Owl             124740
Lophostrix cristata_Crested Owl                     70000
Myiothlypis fulvicauda_Buff-rumped Warbler          65142
Habia atrimaxillaris_Black-cheeked Ant-Tanager      53683
Myiornis atricapillus_Black-capped Pygmy-Tyrant     32097
Thamnophilus bridgesi_Black-hooded Antshrike        18627
Patagioenas nigrirostris_Short-billed Pigeon        17507
Microrhopias quixensis_Dot-winged Antwren           16940
Tinamus major_Great Tinamou                         15547
Name: count, dtype: int64


In [32]:
# Train model with only sentinel species
X_sentinel = df[['is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant']].copy()
y = df['is_simulation'].copy()

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_sentinel, y, test_size=0.2, random_state=42, stratify=y
)

xgb_sentinel = xgb.XGBClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    scale_pos_weight=20,
    eval_metric='logloss'
)

xgb_sentinel.fit(X_train_s, y_train_s)
y_pred_proba_sentinel = xgb_sentinel.predict_proba(X_test_s)[:, 1]

print(f"Sentinel species model ROC-AUC: {roc_auc_score(y_test_s, y_pred_proba_sentinel):.4f}")

# Find best threshold
thresholds = np.arange(0.1, 0.9, 0.05)
best_f1_s = 0
best_thresh_s = 0.5

for thresh in thresholds:
    y_pred = (y_pred_proba_sentinel >= thresh).astype(int)
    f1 = f1_score(y_test_s, y_pred)
    if f1 > best_f1_s:
        best_f1_s = f1
        best_thresh_s = thresh

y_pred_sentinel = (y_pred_proba_sentinel >= best_thresh_s).astype(int)

print(f"Best threshold: {best_thresh_s:.2f}, F1: {best_f1_s:.3f}")
print(classification_report(y_test_s, y_pred_sentinel))

print(f"\nFeature importance:")
sentinel_features = ['is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant']
for i, feature in enumerate(sentinel_features):
    print(f"{feature}: {xgb_sentinel.feature_importances_[i]:.3f}")

Sentinel species model ROC-AUC: 0.5989
Best threshold: 0.15, F1: 0.105
              precision    recall  f1-score   support

           0       0.99      0.97      0.98    152542
           1       0.07      0.22      0.11      1331

    accuracy                           0.97    153873
   macro avg       0.53      0.60      0.54    153873
weighted avg       0.99      0.97      0.98    153873


Feature importance:
is_Orange_billed_Sparrow: 0.692
is_Antpitta: 0.308
is_Pygmy_Tyrant: 0.000


In [30]:
combined_features = core_features + ['is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant']

X_combined = df[combined_features].copy()
y = df['is_simulation'].copy()

X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

# Scale only acoustic features
scaler_cb = StandardScaler()
X_train_cb_scaled = X_train_cb.copy()
X_test_cb_scaled = X_test_cb.copy()

X_train_cb_scaled[core_features] = scaler_cb.fit_transform(X_train_cb[core_features])
X_test_cb_scaled[core_features] = scaler_cb.transform(X_test_cb[core_features])

# Use XGBoost
xgb_cb = xgb.XGBClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=20,
    eval_metric='logloss'
)

xgb_cb.fit(X_train_cb_scaled, y_train_cb)
y_pred_cb = xgb_cb.predict(X_test_cb_scaled)
y_pred_proba_cb = xgb_cb.predict_proba(X_test_cb_scaled)[:, 1]

print(f"Combined model ROC-AUC: {roc_auc_score(y_test_cb, y_pred_proba_cb):.4f}")
print(classification_report(y_test_cb, y_pred_cb))

# Check feature importance
importance = pd.DataFrame({
    'feature': combined_features,
    'importance': xgb_cb.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature importance:")
print(importance)

Combined model ROC-AUC: 0.9460
              precision    recall  f1-score   support

           0       1.00      0.98      0.99    152542
           1       0.16      0.48      0.24      1331

    accuracy                           0.97    153873
   macro avg       0.58      0.73      0.61    153873
weighted avg       0.99      0.97      0.98    153873


Feature importance:
                     feature  importance
0                 RMS Energy    0.203435
10  is_Orange_billed_Sparrow    0.138208
8                    MFCC_12    0.131354
7                     MFCC_9    0.110180
2          Spectral Flatness    0.107376
11               is_Antpitta    0.079221
6                     MFCC_8    0.046717
3         Spectral Bandwidth    0.041211
12           is_Pygmy_Tyrant    0.039991
4     Spectral Rolloff (85%)    0.035542
1          Spectral Contrast    0.034081
9                    MFCC_13    0.018895
5             Onset Strength    0.013788


In [31]:
thresholds = np.arange(0.1, 0.9, 0.05)
results = []

for thresh in thresholds:
    y_pred = (y_pred_proba_cb >= thresh).astype(int)
    precision = precision_score(y_test_cb, y_pred)
    recall = recall_score(y_test_cb, y_pred)
    f1 = f1_score(y_test_cb, y_pred)
    results.append([thresh, precision, recall, f1])

results_df = pd.DataFrame(results, columns=['Threshold', 'Precision', 'Recall', 'F1'])

# Find best F1 threshold
best_f1_row = results_df.loc[results_df['F1'].idxmax()]
print(f"Best F1 threshold: {best_f1_row['Threshold']:.2f}")
print(f"Precision: {best_f1_row['Precision']:.3f}, Recall: {best_f1_row['Recall']:.3f}, F1: {best_f1_row['F1']:.3f}")

# Show top 5 thresholds
print("\nTop 5 thresholds by F1:")
print(results_df.nlargest(5, 'F1').round(3))

best_thresh = best_f1_row['Threshold']
y_pred_best = (y_pred_proba_cb >= best_thresh).astype(int)

print(f"\nResults with threshold {best_thresh:.2f}:")
print(classification_report(y_test_cb, y_pred_best))

Best F1 threshold: 0.65
Precision: 0.283, Recall: 0.292, F1: 0.287

Top 5 thresholds by F1:
    Threshold  Precision  Recall     F1
11       0.65      0.283   0.292  0.287
10       0.60      0.242   0.340  0.283
12       0.70      0.334   0.239  0.279
9        0.55      0.198   0.408  0.266
13       0.75      0.408   0.188  0.257

Results with threshold 0.65:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    152542
           1       0.28      0.29      0.29      1331

    accuracy                           0.99    153873
   macro avg       0.64      0.64      0.64    153873
weighted avg       0.99      0.99      0.99    153873



Combined 10 acoustic features (RMS Energy, MFCCs, spectral characteristics) with 3 sentinel species indicators (Orange-billed Sparrow, Streak-chested Antpitta, Black-capped Pygmy-Tyrant).
Results: ROC-AUC of 0.946 with optimal threshold 0.65 achieving 28.3% precision, 29.2% recall, and F1-score of 0.287. RMS Energy ranked as top feature, followed by Orange-billed Sparrow presence. Sentinel species improved performance over acoustic features alone.

In [ ]:
# Use expanded features to find overall importance
expanded_features = core_features + [
    'is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant',
    'Temperature', 'Humidity', 'Windspeed', 'Datetime_hour',
    'Human Activity Score', 'confidence'
]

X_expanded = df[expanded_features].copy()
y = df['is_simulation'].copy()

X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
    X_expanded, y, test_size=0.2, random_state=42, stratify=y
)

scaler_exp = StandardScaler()
features_to_scale = [f for f in expanded_features if f not in ['is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant']]

X_train_exp_scaled = X_train_exp.copy()
X_test_exp_scaled = X_test_exp.copy()
X_train_exp_scaled[features_to_scale] = scaler_exp.fit_transform(X_train_exp[features_to_scale])
X_test_exp_scaled[features_to_scale] = scaler_exp.transform(X_test_exp[features_to_scale])

xgb_exp = xgb.XGBClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=20,
    eval_metric='logloss'
)

xgb_exp.fit(X_train_exp_scaled, y_train_exp)

importance_exp = pd.DataFrame({
    'feature': expanded_features,
    'importance': xgb_exp.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature importance (all expanded features):")
print(importance_exp)

Feature importance (all expanded features):
                     feature  importance
15                 Windspeed    0.217334
13               Temperature    0.154700
16             Datetime_hour    0.107215
10  is_Orange_billed_Sparrow    0.078304
0                 RMS Energy    0.061010
14                  Humidity    0.055978
8                    MFCC_12    0.055621
2          Spectral Flatness    0.044546
3         Spectral Bandwidth    0.042133
12           is_Pygmy_Tyrant    0.036325
6                     MFCC_8    0.033549
7                     MFCC_9    0.029982
9                    MFCC_13    0.029078
4     Spectral Rolloff (85%)    0.020125
1          Spectral Contrast    0.014162
18                confidence    0.010001
11               is_Antpitta    0.006065
5             Onset Strength    0.003870
17      Human Activity Score    0.000000


In [35]:
# Use sentinel species + top 6 features
custom_features = [
    'is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant',
    'Windspeed', 'Temperature', 'Datetime_hour', 
    'RMS Energy', 'Humidity', 'MFCC_12'
]

print("Custom features:", custom_features)

X_custom = df[custom_features].copy()
y = df['is_simulation'].copy()

X_train_custom, X_test_custom, y_train_custom, y_test_custom = train_test_split(
    X_custom, y, test_size=0.2, random_state=42, stratify=y
)

scaler_custom = StandardScaler()
features_to_scale_custom = [f for f in custom_features if not f.startswith('is_')]

X_train_custom_scaled = X_train_custom.copy()
X_test_custom_scaled = X_test_custom.copy()
X_train_custom_scaled[features_to_scale_custom] = scaler_custom.fit_transform(X_train_custom[features_to_scale_custom])
X_test_custom_scaled[features_to_scale_custom] = scaler_custom.transform(X_test_custom[features_to_scale_custom])

xgb_custom = xgb.XGBClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=20,
    eval_metric='logloss'
)

xgb_custom.fit(X_train_custom_scaled, y_train_custom)
y_pred_proba_custom = xgb_custom.predict_proba(X_test_custom_scaled)[:, 1]

print(f"\nCustom model ROC-AUC: {roc_auc_score(y_test_custom, y_pred_proba_custom):.4f}")

thresholds = np.arange(0.1, 0.9, 0.05)
best_f1_custom = 0
best_thresh_custom = 0.5

for thresh in thresholds:
    y_pred = (y_pred_proba_custom >= thresh).astype(int)
    f1 = f1_score(y_test_custom, y_pred)
    if f1 > best_f1_custom:
        best_f1_custom = f1
        best_thresh_custom = thresh

y_pred_custom = (y_pred_proba_custom >= best_thresh_custom).astype(int)
print(f"Best threshold: {best_thresh_custom:.2f}, F1: {best_f1_custom:.3f}")
print(classification_report(y_test_custom, y_pred_custom))

Custom features: ['is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant', 'Windspeed', 'Temperature', 'Datetime_hour', 'RMS Energy', 'Humidity', 'MFCC_12']

Custom model ROC-AUC: 0.9897
Best threshold: 0.80, F1: 0.473
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    152542
           1       0.39      0.59      0.47      1331

    accuracy                           0.99    153873
   macro avg       0.69      0.79      0.73    153873
weighted avg       0.99      0.99      0.99    153873



Enhanced feature selection combining sentinel species, acoustic, environmental, and temporal features improved simulation detection performance. Combined 3 sentinel species indicators with 6 key features: Windspeed, Temperature, Datetime_hour, RMS Energy, Humidity, and MFCC_12.

Results: ROC-AUC of 0.990 with optimal threshold 0.80 achieving 39% precision, 59% recall, and 
F1-score of 0.473. 

Key Finding: Environmental conditions (weather, temperature) and temporal patterns (hour of day) are critical predictors.

In [36]:
# Filter data to only include sentinel species detections
sentinel_mask = (df['is_Orange_billed_Sparrow'] == 1) | (df['is_Antpitta'] == 1) | (df['is_Pygmy_Tyrant'] == 1)
df_sentinel_only = df[sentinel_mask].copy()

print(f"Original dataset: {len(df)} rows")
print(f"Sentinel species only: {len(df_sentinel_only)} rows")
print(f"Reduction: {(1 - len(df_sentinel_only)/len(df))*100:.1f}%")

print(f"\nSimulation distribution:")
print(f"Original - Simulations: {df['is_simulation'].sum()}, Total: {len(df)}")
print(f"Filtered - Simulations: {df_sentinel_only['is_simulation'].sum()}, Total: {len(df_sentinel_only)}")

sim_rate_original = df['is_simulation'].mean()
sim_rate_filtered = df_sentinel_only['is_simulation'].mean()
print(f"\nSimulation rate:")
print(f"Original: {sim_rate_original:.4f} ({sim_rate_original*100:.2f}%)")
print(f"Filtered: {sim_rate_filtered:.4f} ({sim_rate_filtered*100:.2f}%)")
print(f"Rate increase: {sim_rate_filtered/sim_rate_original:.2f}x")

Original dataset: 769365 rows
Sentinel species only: 53643 rows
Reduction: 93.0%

Simulation distribution:
Original - Simulations: 6657, Total: 769365
Filtered - Simulations: 1659, Total: 53643

Simulation rate:
Original: 0.0087 (0.87%)
Filtered: 0.0309 (3.09%)
Rate increase: 3.57x


In [37]:
custom_features = [
    'is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant',
    'Windspeed', 'Temperature', 'Datetime_hour', 
    'RMS Energy', 'Humidity', 'MFCC_12'
]

X_filtered = df_sentinel_only[custom_features].copy()
y_filtered = df_sentinel_only['is_simulation'].copy()

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

print(f"Filtered train set: {len(X_train_f)} rows, Simulations: {y_train_f.sum()}")
print(f"Filtered test set: {len(X_test_f)} rows, Simulations: {y_test_f.sum()}")

scaler_f = StandardScaler()
features_to_scale = [f for f in custom_features if not f.startswith('is_')]

X_train_f_scaled = X_train_f.copy()
X_test_f_scaled = X_test_f.copy()
X_train_f_scaled[features_to_scale] = scaler_f.fit_transform(X_train_f[features_to_scale])
X_test_f_scaled[features_to_scale] = scaler_f.transform(X_test_f[features_to_scale])
pos_weight = (y_train_f == 0).sum() / y_train_f.sum()
print(f"New scale_pos_weight: {pos_weight:.1f}")

xgb_filtered = xgb.XGBClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=pos_weight,
    eval_metric='logloss'
)

xgb_filtered.fit(X_train_f_scaled, y_train_f)
y_pred_proba_f = xgb_filtered.predict_proba(X_test_f_scaled)[:, 1]

print(f"\nFiltered model ROC-AUC: {roc_auc_score(y_test_f, y_pred_proba_f):.4f}")

thresholds = np.arange(0.1, 0.9, 0.05)
best_f1_f = 0
best_thresh_f = 0.5

for thresh in thresholds:
    y_pred = (y_pred_proba_f >= thresh).astype(int)
    f1 = f1_score(y_test_f, y_pred)
    if f1 > best_f1_f:
        best_f1_f = f1
        best_thresh_f = thresh

y_pred_f = (y_pred_proba_f >= best_thresh_f).astype(int)
print(f"Best threshold: {best_thresh_f:.2f}, F1: {best_f1_f:.3f}")
print(classification_report(y_test_f, y_pred_f))

Filtered train set: 42914 rows, Simulations: 1327
Filtered test set: 10729 rows, Simulations: 332
New scale_pos_weight: 31.3

Filtered model ROC-AUC: 0.9908
Best threshold: 0.85, F1: 0.675
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     10397
           1       0.54      0.89      0.68       332

    accuracy                           0.97     10729
   macro avg       0.77      0.93      0.83     10729
weighted avg       0.98      0.97      0.98     10729



Filtering the dataset to include only time periods with sentinel species detections improved model performance, achieving F1-score of 0.675 with 89% recall and 54% precision. I Analyzed only audio segments containing Orange-billed Sparrow, Antpitta, or Pygmy-Tyrant detections, reducing dataset from 769k to 54k rows while retaining 25% of simulation events. The filtered approach increased simulation detection rate from 0.87% to 3.09%, improving F1-score by 43% compared to the full dataset model.

Why it works: Filtering to sentinel species detection periods removes noise from irrelevant time segments and creates a more balanced dataset where human disturbances are 3.6x more likely to occur, enabling the model to learn stronger patterns between environmental conditions and disturbance events.

Potential Concerns: the filtered approach may introduce selection bias by missing disturbances when sentinel species are naturally absent, potentially underestimating disturbance rates. Additionally, temporal correlations between species activity and environmental conditions (temperature, time of day) that also drive human activity patterns could create confounding variables, artificially inflating model performance. The high recall (89%) only applies within sentinel species detection periods, so true system-wide disturbance detection rates remain unknown.

## MLP neural network

### 3 sentinel species + 6 high ranking features USING filtered data (only sentinel detection clips)

In [39]:
X_train_mlp = X_train_f_scaled.copy()
X_test_mlp = X_test_f_scaled.copy()
y_train_mlp = y_train_f.copy()
y_test_mlp = y_test_f.copy()

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.01,
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

mlp_model.fit(X_train_mlp, y_train_mlp)
y_pred_proba_mlp = mlp_model.predict_proba(X_test_mlp)[:, 1]

print(f"MLP ROC-AUC: {roc_auc_score(y_test_mlp, y_pred_proba_mlp):.4f}")

thresholds = np.arange(0.1, 0.9, 0.05)
best_f1_mlp = 0
best_thresh_mlp = 0.5

for thresh in thresholds:
    y_pred = (y_pred_proba_mlp >= thresh).astype(int)
    f1 = f1_score(y_test_mlp, y_pred)
    if f1 > best_f1_mlp:
        best_f1_mlp = f1
        best_thresh_mlp = thresh

y_pred_mlp = (y_pred_proba_mlp >= best_thresh_mlp).astype(int)
print(f"Best threshold: {best_thresh_mlp:.2f}, F1: {best_f1_mlp:.3f}")
print(classification_report(y_test_mlp, y_pred_mlp))

MLP ROC-AUC: 0.9898
Best threshold: 0.40, F1: 0.747
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     10397
           1       0.75      0.75      0.75       332

    accuracy                           0.98     10729
   macro avg       0.87      0.87      0.87     10729
weighted avg       0.98      0.98      0.98     10729



### 3 sentinel species + 6 high ranking features USING full data

In [40]:
custom_features = [
    'is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant',
    'Windspeed', 'Temperature', 'Datetime_hour', 
    'RMS Energy', 'Humidity', 'MFCC_12'
]

# Use original full dataset
X_full_mlp = df[custom_features].copy()
y_full_mlp = df['is_simulation'].copy()

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full_mlp, y_full_mlp, test_size=0.2, random_state=42, stratify=y_full_mlp
)

scaler_full = StandardScaler()
features_to_scale = [f for f in custom_features if not f.startswith('is_')]

X_train_full_scaled = X_train_full.copy()
X_test_full_scaled = X_test_full.copy()
X_train_full_scaled[features_to_scale] = scaler_full.fit_transform(X_train_full[features_to_scale])
X_test_full_scaled[features_to_scale] = scaler_full.transform(X_test_full[features_to_scale])

print(f"Full dataset - Train: {len(X_train_full)}, Test: {len(X_test_full)}")
print(f"Simulations - Train: {y_train_full.sum()}, Test: {y_test_full.sum()}")

# Train MLP on full data
mlp_full = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.01,
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

mlp_full.fit(X_train_full_scaled, y_train_full)
y_pred_proba_full = mlp_full.predict_proba(X_test_full_scaled)[:, 1]

print(f"\nMLP Full Data ROC-AUC: {roc_auc_score(y_test_full, y_pred_proba_full):.4f}")

# Find best threshold
thresholds = np.arange(0.1, 0.9, 0.05)
best_f1_full = 0
best_thresh_full = 0.5

for thresh in thresholds:
    y_pred = (y_pred_proba_full >= thresh).astype(int)
    f1 = f1_score(y_test_full, y_pred)
    if f1 > best_f1_full:
        best_f1_full = f1
        best_thresh_full = thresh

y_pred_full = (y_pred_proba_full >= best_thresh_full).astype(int)
print(f"Best threshold: {best_thresh_full:.2f}, F1: {best_f1_full:.3f}")
print(classification_report(y_test_full, y_pred_full))

Full dataset - Train: 615492, Test: 153873
Simulations - Train: 5326, Test: 1331

MLP Full Data ROC-AUC: 0.9877
Best threshold: 0.20, F1: 0.486
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    152542
           1       0.51      0.46      0.49      1331

    accuracy                           0.99    153873
   macro avg       0.75      0.73      0.74    153873
weighted avg       0.99      0.99      0.99    153873



### 3 sentinel species + 6 high ranking features + acoustic features USING full data 

In [41]:
expanded_features = [
    'RMS Energy', 'Spectral Contrast', 'Spectral Flatness',
    'Spectral Bandwidth', 'Spectral Rolloff (85%)', 'Onset Strength', 
    'MFCC_8', 'MFCC_9', 'MFCC_12', 'MFCC_13',
    'is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant',
    'Temperature', 'Humidity', 'Windspeed', 'Datetime_hour',
    'Human Activity Score', 'confidence'
]

print(f"Total features: {len(expanded_features)}")
print("Features:", expanded_features)

X_exp = df[expanded_features].copy()
y_exp = df['is_simulation'].copy()

X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
    X_exp, y_exp, test_size=0.2, random_state=42, stratify=y_exp
)

# Scale features
scaler_exp = StandardScaler()
features_to_scale_exp = [f for f in expanded_features if not f.startswith('is_')]

X_train_exp_scaled = X_train_exp.copy()
X_test_exp_scaled = X_test_exp.copy()
X_train_exp_scaled[features_to_scale_exp] = scaler_exp.fit_transform(X_train_exp[features_to_scale_exp])
X_test_exp_scaled[features_to_scale_exp] = scaler_exp.transform(X_test_exp[features_to_scale_exp])

mlp_exp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.01,
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

mlp_exp.fit(X_train_exp_scaled, y_train_exp)
y_pred_proba_exp = mlp_exp.predict_proba(X_test_exp_scaled)[:, 1]

print(f"\nMLP {len(expanded_features)} Features ROC-AUC: {roc_auc_score(y_test_exp, y_pred_proba_exp):.4f}")

thresholds = np.arange(0.1, 0.9, 0.05)
best_f1_exp = 0
best_thresh_exp = 0.5

for thresh in thresholds:
    y_pred = (y_pred_proba_exp >= thresh).astype(int)
    f1 = f1_score(y_test_exp, y_pred)
    if f1 > best_f1_exp:
        best_f1_exp = f1
        best_thresh_exp = thresh

y_pred_exp = (y_pred_proba_exp >= best_thresh_exp).astype(int)
print(f"Best threshold: {best_thresh_exp:.2f}, F1: {best_f1_exp:.3f}")
print(classification_report(y_test_exp, y_pred_exp))

Total features: 19
Features: ['RMS Energy', 'Spectral Contrast', 'Spectral Flatness', 'Spectral Bandwidth', 'Spectral Rolloff (85%)', 'Onset Strength', 'MFCC_8', 'MFCC_9', 'MFCC_12', 'MFCC_13', 'is_Orange_billed_Sparrow', 'is_Antpitta', 'is_Pygmy_Tyrant', 'Temperature', 'Humidity', 'Windspeed', 'Datetime_hour', 'Human Activity Score', 'confidence']

MLP 19 Features ROC-AUC: 0.9983
Best threshold: 0.40, F1: 0.766
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    152542
           1       0.76      0.77      0.77      1331

    accuracy                           1.00    153873
   macro avg       0.88      0.88      0.88    153873
weighted avg       1.00      1.00      1.00    153873



The multi-layer perceptron using 19 features achieved optimal simulation detection with F1-score of 0.766, 76% precision, 77% recall, and ROC-AUC of 0.998. The comprehensive feature set combined acoustic characteristics, sentinel species indicators, environmental conditions, temporal patterns, and detection metadata. This approach outperformed both gradient boosting methods and data filtering strategies

In [ ]:
# Check training vs test performance
y_train_pred_proba = mlp_exp.predict_proba(X_train_exp_scaled)[:, 1]
y_train_pred = (y_train_pred_proba >= best_thresh_exp).astype(int)

print("Training set performance:")
train_f1 = f1_score(y_train_exp, y_train_pred)
train_precision = precision_score(y_train_exp, y_train_pred)
train_recall = recall_score(y_train_exp, y_train_pred)
train_auc = roc_auc_score(y_train_exp, y_train_pred_proba)

print(f"Train - F1: {train_f1:.3f}, Precision: {train_precision:.3f}, Recall: {train_recall:.3f}, AUC: {train_auc:.4f}")
print(f"Test  - F1: {best_f1_exp:.3f}, Precision: {0.76:.3f}, Recall: {0.77:.3f}, AUC: {roc_auc_score(y_test_exp, y_pred_proba_exp):.4f}")

print(f"\nPerformance gap:")
print(f"F1 gap: {train_f1 - best_f1_exp:.3f}")
print(f"AUC gap: {train_auc - roc_auc_score(y_test_exp, y_pred_proba_exp):.4f}")

print(f"\nModel info:")
print(f"Features: {len(expanded_features)}")
print(f"Hidden layers: {mlp_exp.hidden_layer_sizes}")
print(f"Total parameters: ~{sum(mlp_exp.hidden_layer_sizes) * len(expanded_features)}")
print(f"Training samples: {len(y_train_exp)}")
print(f"Parameters/samples ratio: {(sum(mlp_exp.hidden_layer_sizes) * len(expanded_features))/len(y_train_exp):.6f}")

Training set performance:
Train - F1: 0.794, Precision: 0.791, Recall: 0.798, AUC: 0.9987
Test  - F1: 0.766, Precision: 0.760, Recall: 0.770, AUC: 0.9983

Performance gap:
F1 gap: 0.028
AUC gap: 0.0004

Model info:
Features: 19
Hidden layers: (128, 64, 32)
Total parameters: ~4256
Training samples: 615492
Parameters/samples ratio: 0.006915


### Overfitting?

The model shows minimal signs of overfitting with only small performance gaps between training and test sets: F1 difference of 0.028 (79.4% vs 76.6%) and AUC difference of 0.0004 (0.9987 vs 0.9983). 

## Model Performance Comparison Summary

The best performing method was MLP with 19 features (F1=0.766, 76% precision, 77% recall, ROC-AUC=0.998)

**Performance Ranking**
1. MLP + 19 features (full dataset): F1=0.766
2. MLP + 9 features (filtered data): F1=0.747  
3. XGBoost + 9 features (filtered data): F1=0.675
4. XGBoost + 9 features (full dataset): F1=0.473
5. Sentinel species only: F1=0.105

**Sentinel Species Impact**
Highly valuable - Orange-billed Sparrow consistently ranked as 2nd-4th most important feature across models. Sentinel species improved all models compared to acoustic-only approaches.

**MLP Advantages**
Good handling of complex non-linear relationships between diverse feature types. Balanced precision/recall with minimal overfitting risk

**Concerns?**
Performance may not transfer to different ecosystems or species compositions?

**Next Steps?**
Cross-validation across different geographic locations/seasons? Further fine-tuning and try out different deep learning/nlp models.